# 18 · Wallet — pruebas

Este notebook **prueba la clase `binpan.Wallet`** (balances spot/margin, snapshots y performance) sin
necesitar claves de API reales.

`Wallet` solo habla con endpoints **firmados** de Binance (cuenta privada). Para poder ejecutarlo en
cualquier máquina y en CI, aquí **interceptamos la frontera de red/credenciales** —el
`signed_request` de `panzer.BinanceClient`— y devolvemos respuestas de ejemplo con la forma exacta de
la API. Así se ejercita todo el cableado de BinPan (parseo a DataFrame, filtrado, conversión de tipos,
caché de snapshots y cálculo de performance) de forma determinista.

> Para el uso **real** con tus credenciales, mira la última sección y el notebook
> `17_credentials_and_panzer`.


## 1) Respuestas de API simuladas

Forma idéntica a la de Binance: `/api/v3/account` (spot), `/sapi/v1/margin/account` (margin cross) y
`/sapi/v1/accountSnapshot` (snapshots diarios). Incluimos assets a cero para comprobar el filtrado.


In [ ]:
from unittest.mock import patch
import pandas as pd

FAKE_RESPONSES = {
    "/api/v3/account": {
        "accountType": "SPOT",
        "balances": [
            {"asset": "BTC",  "free": "0.50000000", "locked": "0.10000000"},
            {"asset": "BNB",  "free": "2.00000000", "locked": "0.00000000"},
            {"asset": "USDT", "free": "1000.00000000", "locked": "0.00000000"},
            {"asset": "XRP",  "free": "0.00000000", "locked": "0.00000000"},  # vacio -> se filtra
        ],
    },
    "/sapi/v1/margin/account": {
        "totalAssetOfBtc": "6.82",
        "userAssets": [
            {"asset": "BTC", "borrowed": "0.0", "free": "0.004", "interest": "0.0", "locked": "0.0", "netAsset": "0.004"},
            {"asset": "BNB", "borrowed": "201.6", "free": "2346.5", "interest": "0.0", "locked": "0.0", "netAsset": "2144.8"},
            {"asset": "ETH", "borrowed": "0.0", "free": "0.0", "interest": "0.0", "locked": "0.0", "netAsset": "0.0"},  # todo 0 -> se filtra
        ],
    },
    "/sapi/v1/accountSnapshot": {
        "snapshotVos": [
            {"updateTime": 1660000000000, "data": {"totalAssetOfBtc": "6.50",
                "balances":   [{"asset": "BTC", "free": "0.5", "locked": "0.0"}],
                "userAssets": [{"asset": "BTC", "free": "0.5", "locked": "0.0"}]}},
            {"updateTime": 1660086400000, "data": {"totalAssetOfBtc": "7.00",
                "balances":   [{"asset": "BTC", "free": "0.6", "locked": "0.0"}],
                "userAssets": [{"asset": "BTC", "free": "0.6", "locked": "0.0"}]}},
        ],
    },
}

def fake_signed_request(self, method, endpoint, params=None, recv_window=10000, sign=True):
    """Sustituye la llamada firmada real por una respuesta de ejemplo."""
    assert endpoint in FAKE_RESPONSES, f"endpoint inesperado: {endpoint}"
    return FAKE_RESPONSES[endpoint]

# convert_coin consultaria precios publicos por red; lo fijamos a una tasa determinista (1 BTC = 60000).
def fake_convert_coin(coin, convert_to, coin_qty, decimal_mode, prices=None):
    rate = {"BTC": 60000.0, "BNB": 600.0, "USDT": 1.0}.get(coin, 1.0)
    quote = {"BUSD": 1.0, "USDT": 1.0, "BTC": 60000.0}.get(convert_to, 1.0)
    return float(coin_qty) * rate / quote

print("mocks listos")

## 2) Construcción y balances

`Wallet()` llama en el `__init__` a `update_spot()` y `update_margin()`. Verificamos que:
- se construye sin credenciales reales (gracias al mock),
- los balances se parsean a DataFrame,
- los assets vacíos (spot) y a cero (margin) se filtran.


In [ ]:
with patch("panzer.BinanceClient.signed_request", fake_signed_request), \
     patch("binpan.wallet.convert_coin", fake_convert_coin):
    import binpan
    w = binpan.Wallet(time_zone="Europe/Madrid")

print("binpan", binpan.__version__)
print("\nSPOT balances:")
print(w.spot)
print("\nMARGIN balances:")
print(w.margin)

assert list(w.spot.index) == ["BTC", "BNB", "USDT"], "XRP vacio deberia filtrarse"
assert w.spot.loc["BTC", "free"] == 0.5 and w.spot.loc["BTC", "locked"] == 0.1
assert set(w.margin.index) == {"BTC", "BNB"}, "ETH a cero deberia filtrarse"
print("\nOK: balances spot/margin parseados y vacios filtrados")

## 3) Snapshots (regresión)

`update_spot_snapshot()` / `update_margin_snapshot()` piden el snapshot diario y lo guardan en
`self.spot_snapshot` / `self.margin_snapshot`.

> **Regresión cubierta:** antes estos métodos se llamaban `spot_snapshot`/`margin_snapshot`, **el mismo
> nombre** que un atributo que el `__init__` ponía a `None`. El atributo *sombreaba* al método y
> `w.spot_snapshot(...)` reventaba con `TypeError: 'NoneType' object is not callable`. El test comprueba
> que ahora son **invocables** y que pueblan el atributo con un DataFrame.


In [ ]:
with patch("panzer.BinanceClient.signed_request", fake_signed_request), \
     patch("binpan.wallet.convert_coin", fake_convert_coin):
    spot_snap = w.update_spot_snapshot(snapshot_days=7)
    margin_snap = w.update_margin_snapshot(snapshot_days=7)

# los metodos son invocables (no sombreados por un atributo None)
assert callable(type(w).update_spot_snapshot)
assert callable(type(w).update_margin_snapshot)

# devuelven el snapshot y pueblan el atributo correspondiente
assert isinstance(spot_snap, pd.DataFrame) and not spot_snap.empty
assert isinstance(w.spot_snapshot, pd.DataFrame)
assert isinstance(margin_snap, pd.DataFrame) and not margin_snap.empty
assert isinstance(w.margin_snapshot, pd.DataFrame)
assert w.spot_requested_days == 7 and w.margin_requested_days == 7

print(spot_snap)
print("\nOK: snapshots invocables y atributos poblados con DataFrame")

## 4) Performance + invariante de no-pisado

`spot_wallet_performance()` / `margin_wallet_performance()` calculan la diferencia de `totalAssetOfBtc`
entre el primer y el último día del snapshot, opcionalmente convertida a otra moneda.

> **Regresión cubierta:** antes estas funciones *machacaban* `self.spot`/`self.margin` (los balances)
> con las filas del snapshot, y si el caché no se refrescaba (mismos `days`) leían
> `self.spot['totalAssetOfBtc']` sobre los balances → `KeyError`. Ahora usan el atributo snapshot como
> caché y **no tocan los balances**.


In [ ]:
with patch("panzer.BinanceClient.signed_request", fake_signed_request), \
     patch("binpan.wallet.convert_coin", fake_convert_coin):
    # mismos days que el snapshot ya cacheado (este era el caso del KeyError)
    perf_btc  = w.spot_wallet_performance(decimal_mode=False, days=7, convert_to="BTC")
    perf_busd = w.spot_wallet_performance(decimal_mode=False, days=7, convert_to="BUSD")
    mperf_btc = w.margin_wallet_performance(decimal_mode=False, days=7, convert_to="BTC")

print("spot performance  BTC :", perf_btc, " (esperado 7.00 - 6.50 = 0.5)")
print("spot performance  BUSD:", perf_busd, " (esperado 0.5 * 60000 = 30000)")
print("margin performance BTC:", mperf_btc, " (esperado 0.5)")

assert abs(perf_btc - 0.5) < 1e-9
assert abs(perf_busd - 0.5 * 60000.0) < 1e-6
assert abs(mperf_btc - 0.5) < 1e-9

# invariante: los balances NO se han pisado con el snapshot
assert list(w.spot.index) == ["BTC", "BNB", "USDT"], "self.spot fue pisado!"
assert set(w.margin.index) == {"BTC", "BNB"}, "self.margin fue pisado!"
print("\nOK: performance correcta y balances intactos")

## 5) Resumen

Todo lo anterior corre **sin claves** porque mockeamos `panzer.BinanceClient.signed_request`. Si llegamos
aquí sin `AssertionError`, la clase `Wallet` está sana tras la migración de credenciales a panzer.


In [ ]:
print("TODOS LOS TESTS DE Wallet PASARON")

## 6) Uso real (opcional, requiere credenciales)

Con `api_key`/`api_secret` configuradas en panzer (`~/.panzer_creds`; ver `17_credentials_and_panzer`),
el uso real es **sin mocks**:

```python
import binpan

w = binpan.Wallet(time_zone="Europe/Madrid")   # panzer pedira las claves la 1a vez
w.spot       # DataFrame de balances spot
w.margin     # DataFrame de balances margin cross

w.update_spot_snapshot(snapshot_days=7)         # snapshot diario -> w.spot_snapshot
w.spot_wallet_performance(decimal_mode=False, days=7, convert_to="USDT")
```

La celda siguiente está **desactivada** (`if False:`) para que el notebook siga corriendo en CI sin
claves. Cambia a `if True:` para probar contra tu cuenta real.


In [ ]:
if False:  # cambia a True con credenciales configuradas en panzer
    import binpan
    real = binpan.Wallet(time_zone="Europe/Madrid")
    print(real.spot)
    print(real.update_spot_snapshot(snapshot_days=7))
    print("performance 7d (USDT):",
          real.spot_wallet_performance(decimal_mode=False, days=7, convert_to="USDT"))